In [ ]:
import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Modern LangChain imports
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationTokenBufferMemory,
    ConversationSummaryBufferMemory
)

_ = load_dotenv(find_dotenv())
warnings.filterwarnings('ignore')

llm_model = "gpt-4o-mini"

# Initialize the LLM
llm = ChatOpenAI(temperature=0.0, model=llm_model)

# 1. ConversationBufferMemory

In [ ]:
print("\n--- ConversationBufferMemory ---")
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

print(conversation.invoke({"input": "Hi, my name is Andrew"})["response"])
print(conversation.invoke({"input": "What is 1+1?"})["response"])
print(conversation.invoke({"input": "What is my name?"})["response"])

# Inspecting the memory buffer
print("\nBuffer contents:")
print(memory.buffer)
print("\nMemory variables:")
print(memory.load_memory_variables({}))

# Manually saving context to memory
memory = ConversationBufferMemory()
memory.save_context({"input": "Hi"}, {"output": "What's up"})
print("\nManual memory save:")
print(memory.buffer)
print(memory.load_memory_variables({}))

memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
print(memory.load_memory_variables({}))

In [ ]:
# ==========================================
# 2. ConversationBufferWindowMemory

In [ ]:
# ==========================================
print("\n--- ConversationBufferWindowMemory ---")
# Keeps only the last 'k' interactions
memory = ConversationBufferWindowMemory(k=1)               

memory.save_context({"input": "Hi"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})

print("\nWindow Memory variables (k=1):")
print(memory.load_memory_variables({}))

# Using it in a chain
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

print(conversation.invoke({"input": "Hi, my name is Andrew"})["response"])
print(conversation.invoke({"input": "What is 1+1?"})["response"])
print(conversation.invoke({"input": "What is my name?"})["response"]) # Will forget the name because k=1

In [ ]:
# ==========================================
# 3. ConversationTokenBufferMemory

In [ ]:
# ==========================================
print("\n--- ConversationTokenBufferMemory ---")
# Requires 'tiktoken' package to be installed
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"}, {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

print("\nToken Buffer Memory variables (max_tokens=50):")
print(memory.load_memory_variables({}))

In [ ]:
# ==========================================
# 4. ConversationSummaryBufferMemory

In [ ]:
# ==========================================
print("\n--- ConversationSummaryBufferMemory ---")
schedule = (
    "There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. "
    "9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. "
    "At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. "
    "Be sure to bring your laptop to show the latest LLM demo."
)

memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"}, {"output": schedule})

print("\nSummary Buffer Memory variables before prediction:")
print(memory.load_memory_variables({}))

conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

print(conversation.invoke({"input": "What would be a good demo to show?"})["response"])

print("\nSummary Buffer Memory variables after prediction:")
print(memory.load_memory_variables({}))